# Practice 2 — Phân loại CIFAR-10 bằng Pretrained CNN
## So sánh VGG16 và DenseNet121

Bài thực hành khảo sát hiệu quả của **transfer learning**, **fine-tuning** và **learning rate** trên bài toán phân loại ảnh CIFAR-10.
Hai kiến trúc VGG16 và DenseNet121 được sử dụng với trọng số pretrained ImageNet-1K, thay classifier để dự đoán 10 lớp.

Ba cấu hình được áp dụng thống nhất cho mỗi kiến trúc, tạo thành sáu thí nghiệm.
Các thí nghiệm dùng cùng cách chia dữ liệu, tiền xử lý, seed, optimizer, batch size và ngân sách epoch.

Báo cáo gồm dữ liệu và EDA, tiền xử lý, kiến trúc, thiết kế thí nghiệm, huấn luyện, kết quả và phân tích lỗi.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

workspace = Path.cwd().resolve()
project = workspace if (workspace / 'src' / 'config.py').exists() else workspace / 'practice_2_pretrained_model'
if not (project / 'src' / 'config.py').exists():
    raise FileNotFoundError('Mở notebook từ workspace hoặc thư mục practice_2_pretrained_model.')
sys.path.insert(0, str(project / 'src'))
import config

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None

eda = read_json(project / 'outputs' / 'eda.json')
print('Project:', project)
print('Cấu hình đang chọn:', config.MODEL_NAME, config.EXPERIMENT)


## 1. Dữ liệu và phân tích khám phá

CIFAR-10 gồm 60.000 ảnh RGB kích thước 32×32 thuộc 10 lớp.
Tập train chính thức gồm 50.000 ảnh được chia phân tầng thành **45.000 ảnh train** và **5.000 ảnh validation**, với seed 42.
**10.000 ảnh test** được giữ riêng để đánh giá sau khi chọn cấu hình bằng validation.

EDA kiểm tra số lượng ảnh, phân bố lớp, kích thước, kiểu dữ liệu, thống kê RGB và ảnh trùng chính xác.
Thống kê RGB được tính trên tập train; validation và test không được dùng để ước lượng tham số tiền xử lý.


In [ ]:
if eda is None:
    print('Chưa có EDA report. Chuẩn bị dataset rồi chạy src/1_eda.py ngoài notebook.')
else:
    print('Nguồn:', eda['source'], '| Thời điểm:', eda['created'])
    print('Lớp:', ', '.join(eda['classes']))
    fig, ax = plt.subplots(figsize=(11, 4))
    x = np.arange(len(eda['classes']))
    for offset, (name, split) in enumerate(eda['splits'].items()):
        ax.bar(x + offset * 0.25, split['counts'], width=0.25, label=name)
        print(name, split['count'], 'ảnh | shape:', split['shape'], '| dtype:', split['dtype'])
    ax.set_xticks(x + 0.25, eda['classes'], rotation=35)
    ax.set_ylabel('Số ảnh'); ax.set_title('Phân bố lớp theo split'); ax.legend()
    plt.tight_layout(); plt.show()
    print('RGB train mean:', eda['train_rgb']['mean'])
    print('RGB train std:', eda['train_rgb']['std'])
    print('Nhóm ảnh trùng chính xác:', len(eda['duplicates']))


### 1.1. Ảnh mẫu từng lớp

Mỗi lớp được minh họa bằng một ảnh gốc thuộc tập train, kèm tên lớp và chỉ số ảnh.
Các ảnh cho thấy sự đa dạng về hình dạng, màu sắc và bối cảnh; độ phân giải thấp có thể làm các lớp có đặc điểm gần nhau khó phân biệt.


In [ ]:
from data import load_datasets, load_split

data_present = (project / 'data' / 'cifar-10-batches-py' / 'data_batch_1').exists()
if data_present:
    original, test_data = load_datasets(download=False)
    train_ids, val_ids = load_split(original.targets)
    examples = {}
    for index in train_ids:
        label = original.targets[index]
        if label not in examples:
            examples[label] = index
        if len(examples) == 10:
            break
    fig, axes = plt.subplots(2, 5, figsize=(11, 5))
    for label, ax in enumerate(axes.flat):
        index = examples[label]
        ax.imshow(original.data[index]); ax.set_title(f'{original.classes[label]} — ID {index}')
        ax.axis('off')
    plt.tight_layout(); plt.show()
    del original, test_data
else:
    print('Không có dataset local; bỏ qua ảnh mẫu, không tải tự động.')


## 2. Tiền xử lý dữ liệu

Các ảnh được resize thành **224×224**, chuyển sang tensor trong khoảng [0,1] và chuẩn hóa theo từng kênh màu bằng thống kê ImageNet:

$$x'_{c} = \frac{x_c-\mu_c}{\sigma_c}$$

Trong đó $\mu=(0.485,0.456,0.406)$ và $\sigma=(0.229,0.224,0.225)$ theo thứ tự R, G, B.
Tập train được bổ sung random horizontal flip; validation và test không sử dụng biến đổi ngẫu nhiên.

Chuẩn hóa ImageNet duy trì thang đầu vào phù hợp với pretrained weights. Resize không khôi phục chi tiết đã mất từ ảnh gốc 32×32.
Hình minh họa thể hiện ảnh gốc, ảnh resize, ảnh lật và ảnh sau khi đảo normalization.


In [ ]:
print('Image size:', config.IMAGE_SIZE)
print('ImageNet mean:', config.IMAGENET_MEAN)
print('ImageNet std:', config.IMAGENET_STD)
if eda:
    import base64
    from io import BytesIO
    from PIL import Image
    fig, axes = plt.subplots(1, 4, figsize=(11, 3))
    for ax, name in zip(axes, ('original', 'resized', 'flipped', 'restored')):
        payload = eda['preprocessing'][name].split(',', 1)[1]
        ax.imshow(Image.open(BytesIO(base64.b64decode(payload))))
        ax.set_title(name); ax.axis('off')
    plt.tight_layout(); plt.show()


## 3. Kiến trúc và điều chỉnh mô hình

**VGG16** sử dụng các block convolution 3×3 xếp tuần tự.
**DenseNet121** nối đặc trưng từ các lớp trước trong cùng dense block để tái sử dụng thông tin.
Cả hai sử dụng pretrained weights **ImageNet-1K, IMAGENET1K_V1**, thay head 1.000 lớp bằng classifier 10 lớp.

| Model | Backbone và head | Nhóm mở khi fine-tune |
|---|---|---|
| VGG16 | features → GAP 1×1 → flatten → Linear(512,10) | Ba convolution cuối: features.24/26/28 |
| DenseNet121 | features → ReLU → GAP 1×1 → flatten → Linear(1024,10) | denseblock4 và norm5 |

- **Transfer learning:** freeze backbone, chỉ cập nhật head mới.
- **Fine-tuning:** cập nhật thêm nhóm cuối; các nhóm trước vẫn freeze.
- Running statistics của BatchNorm được giữ cố định. Affine parameters chỉ được học nếu thuộc nhóm mở.

Cell thống kê kiến trúc dùng weights ngẫu nhiên trên CPU; số tham số không phụ thuộc giá trị weights. Đây không phải kết quả đánh giá model đã train.


In [ ]:
from model import PretrainedCNN

example_model = PretrainedCNN(config.MODEL_NAME, fine_tune=config.FINE_TUNE, pretrained=False)
print('Model:', example_model.name)
print('Head:', example_model.head)
print('Nhóm cuối:', example_model.final_blocks)
print('Fine-tune:', example_model.fine_tune)
print('Tổng tham số:', sum(p.numel() for p in example_model.parameters()))
print('Tham số được học:', sum(p.numel() for p in example_model.parameters() if p.requires_grad))
del example_model


## 4. Thiết kế thí nghiệm

Mỗi kiến trúc được huấn luyện với ba cấu hình sau:

| Cấu hình | Backbone | Head | Learning rate | Batch size |
|---|---|---|---:|---:|
| A | Freeze | Linear → 10 | 0.001 | 64 |
| B | Mở nhóm cuối | Linear → 10 | 0.001 | 64 |
| C | Mở nhóm cuối | Linear → 10 | 0.0001 | 64 |

**A so với B** khảo sát tác động của fine-tuning. **B so với C** khảo sát tác động của learning rate.
VGG16 và DenseNet121 được so sánh trong cùng cấu hình.

Điều kiện chung: split seed 42, Adam, CrossEntropyLoss, weight decay 0 và 20 epoch mỗi lượt; không sử dụng scheduler hoặc early stopping.
Mỗi thí nghiệm mới bắt đầu từ pretrained weights, không kế thừa weights đã học từ cấu hình khác.


In [ ]:
for name in config.MODEL_NAMES:
    for experiment, values in config.EXPERIMENTS.items():
        print(name, experiment, values)
print('Epoch/lượt:', config.EPOCHS, '| seed:', config.SEED, '| workers:', config.NUM_WORKERS)


## 5. Quy trình huấn luyện và lưu mô hình

Mỗi epoch gồm hai giai đoạn:

1. **Training:** đặt model ở chế độ train; xóa gradient; forward; tính CrossEntropyLoss; backward; cập nhật tham số bằng Adam.
2. **Validation:** chuyển sang eval và no_grad; tính loss, accuracy mà không cập nhật trọng số.

Loss được lấy trung bình theo số ảnh, accuracy là tỷ lệ dự đoán đúng.
Train/validation loss, accuracy, learning rate và thời gian được ghi sau mỗi epoch, đồng thời theo dõi bằng TensorBoard.

- **best.pt:** checkpoint có validation accuracy cao nhất; nếu hòa, ưu tiên validation loss thấp hơn rồi epoch sớm hơn.
- **last.pt:** trạng thái cuối epoch hoàn tất, gồm weights, optimizer và random state để tiếp tục khi gián đoạn.

Việc huấn luyện được thực hiện bằng script Python; các biểu đồ sau được tổng hợp từ kết quả đã lưu.


## 6. Kết quả huấn luyện

Kết quả được đọc từ cấu hình thực tế, history và checkpoint của từng lượt chạy.
Mỗi lượt được nhận diện bằng model, cấu hình, số epoch đã hoàn tất và thiết bị sử dụng.

Các đường cong thể hiện train/validation loss và accuracy theo epoch.
Lượt chưa hoàn thành được ghi rõ số epoch hiện có và không được xem là kết quả cuối của ngân sách 20 epoch.


In [ ]:
import torch

runs = []
for folder in sorted((project / 'outputs' / 'runs').glob('lan_*')):
    metadata = read_json(folder / 'config.json')
    history = read_json(folder / 'history.json') or []
    if not metadata:
        continue
    settings = metadata['settings']
    epochs = {}
    for filename in ('best.pt', 'last.pt'):
        path = folder / filename
        if path.exists():
            checkpoint = torch.load(path, map_location='cpu', weights_only=False)
            epochs[filename] = checkpoint['epoch']
            del checkpoint
    complete = len(history) == settings['train']['epochs']
    compatible = (settings['model_name'] in config.MODEL_NAMES and
                  settings['model_config'] in config.EXPERIMENTS.values())
    print(folder.name, settings['model_name'], settings['model_config'])
    print('Epoch hoàn tất:', len(history), '/', settings['train']['epochs'], '| checkpoint:', epochs)
    print('Đủ epoch:', complete, '| khớp thiết kế thí nghiệm:', compatible,
          '| device:', metadata['environment']['device'])
    print('Nguồn:', folder / 'history.json')
    if history:
        best = max(history, key=lambda row: (row['val_accuracy'], -row['val_loss'], -row['epoch']))
        print('Best validation:', f"{100 * best['val_accuracy']:.2f}%", '| epoch:', best['epoch'])
    runs.append((folder, settings, history))
if not runs:
    print('Chưa có lịch sử train và cấu hình run để hiển thị kết quả huấn luyện.')


In [ ]:
if runs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for folder, settings, history in runs:
        if not history:
            continue
        epochs = [row['epoch'] for row in history]
        label = f"{folder.name} {settings['model_name']} {settings['experiment']}"
        axes[0].plot(epochs, [row['train_loss'] for row in history], label=label + ' train')
        axes[0].plot(epochs, [row['val_loss'] for row in history], '--', label=label + ' val')
        axes[1].plot(epochs, [100 * row['train_accuracy'] for row in history], label=label + ' train')
        axes[1].plot(epochs, [100 * row['val_accuracy'] for row in history], '--', label=label + ' val')
    for ax, title in zip(axes, ('Loss theo epoch', 'Accuracy (%) theo epoch')):
        ax.set_title(title); ax.set_xlabel('Epoch đã hoàn tất'); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()


## 7. Đánh giá và phân tích lỗi

Các chỉ số đánh giá gồm loss, accuracy, macro-F1 và precision/recall/F1 theo từng lớp.
Confusion matrix chuẩn hóa theo hàng thể hiện tỷ lệ dự đoán của từng lớp thật: trục dọc là nhãn thật, trục ngang là nhãn dự đoán.

Validation được dùng để chọn cấu hình và checkpoint. Test chỉ được đánh giá sau khi khóa lựa chọn của từng model.
Báo cáo kết quả phân biệt rõ tập dữ liệu, checkpoint epoch và model được sử dụng; phần dưới hiển thị các báo cáo đánh giá đã có.


### 7.0. Chạy đánh giá từ checkpoint

Checkpoint được đặt tại `outputs/runs/lan_N/best.pt`, sử dụng định dạng đầy đủ của dự án: `model_state`, `settings`, `classes`, `weights`, `epoch`.
Dataset CIFAR-10 và `outputs/cifar10_split.json` cần có trên máy; split phải khớp SHA-256 đã lưu khi train.
File chỉ chứa state_dict riêng lẻ chưa đủ thông tin để xác định cấu hình thí nghiệm.

Cell sau gọi lại `src/3_evaluate.py`, không huấn luyện lại model:

1. Điền `EVALUATION_RUNS` bằng tên các thư mục checkpoint, ví dụ `['lan_1', 'lan_2']`.
2. Đặt `RUN_EVALUATION=True` để thực thi. Mặc định False chỉ xem các kết quả đã lưu.
3. Đặt `EVALUATE_TEST=True` sau khi đã khóa checkpoint bằng validation; False chỉ đánh giá train/validation.
4. Chạy cell đánh giá, rồi chạy các cell phía dưới để in bảng và biểu đồ.

Mỗi lần thực thi cập nhật báo cáo tương ứng trong thư mục run, gồm xác suất 10 lớp khi đánh giá test.
Cấu hình notebook được áp dụng cho lần chạy này; file config.py và trọng số checkpoint không bị thay đổi.


In [ ]:
import runpy

EVALUATION_RUNS = ['lan_1']
RUN_EVALUATION = False
EVALUATE_TEST = False

if RUN_EVALUATION:
    for run_name in EVALUATION_RUNS:
        checkpoint_path = project / 'outputs' / 'runs' / run_name / 'best.pt'
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f'Không có checkpoint: {checkpoint_path}')
    previous_run, previous_test = config.EVAL_RUN, config.RUN_TEST
    try:
        for run_name in EVALUATION_RUNS:
            config.EVAL_RUN = run_name
            config.RUN_TEST = EVALUATE_TEST
            print('Đánh giá:', run_name, '| gồm test:', EVALUATE_TEST)
            runpy.run_path(str(project / 'src' / '3_evaluate.py'), run_name='__main__')
    finally:
        config.EVAL_RUN, config.RUN_TEST = previous_run, previous_test
else:
    print('Chế độ đọc báo cáo đã lưu; chưa thực thi đánh giá checkpoint.')


In [ ]:
reports = []
for folder in sorted((project / 'outputs' / 'runs').glob('lan_*')):
    for split in ('train', 'validation', 'test'):
        report = read_json(folder / f'{split}_metrics.json')
        if report:
            reports.append(report)
            print(folder.name, report['settings']['model_name'], split, 'epoch', report['epoch'])
            print('Accuracy:', report['accuracy'], '| loss:', report['loss'], '| macro-F1:', report['macro_f1'])
            fig, ax = plt.subplots(figsize=(6, 5))
            image = ax.imshow(report['confusion_normalized'], cmap='Blues', vmin=0, vmax=1)
            ax.set_xticks(range(10), report['classes'], rotation=60)
            ax.set_yticks(range(10), report['classes'])
            ax.set_xlabel('Dự đoán'); ax.set_ylabel('Nhãn thật'); ax.set_title(f'{folder.name} — {split}')
            fig.colorbar(image, ax=ax); plt.tight_layout(); plt.show()
if not reports:
    print('Chưa có báo cáo evaluate. Chưa có số liệu test hoặc kết luận per-class.')


### 7.1. Phân tích sau test: TP, TN, FP, FN và các tỷ lệ

Với mỗi lớp, áp dụng **One-vs-Rest (OvR)**: lớp đang xét là positive, chín lớp còn lại là negative.

| Đại lượng | Ý nghĩa đối với lớp đang xét |
|---|---|
| TP | Ảnh đúng lớp và được dự đoán đúng lớp |
| FN | Ảnh đúng lớp nhưng bị dự đoán sang lớp khác |
| FP | Ảnh thuộc lớp khác nhưng bị dự đoán thành lớp đang xét |
| TN | Ảnh thuộc lớp khác và không bị dự đoán thành lớp đang xét |

$$TPR=Recall=\frac{TP}{TP+FN},\qquad FPR=\frac{FP}{FP+TN},\qquad Specificity=1-FPR$$
$$Precision=\frac{TP}{TP+FP},\qquad F1=\frac{2TP}{2TP+FP+FN}$$

TPR cao thể hiện ít bỏ sót; FPR thấp thể hiện ít nhận nhầm từ lớp khác.
Các tỷ lệ trong bảng dưới dùng **dự đoán argmax**. Mẫu số bằng 0 được ghi NaN (không xác định), không coi là kết quả tốt.
Trên test CIFAR-10 chuẩn, mỗi lớp có 1.000 positive và 9.000 negative; specificity cao riêng lẻ chưa đủ chứng minh model tốt.


In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score


def analyse_test_report(report):
    classes = report['classes']
    records = report['predictions']
    actual = np.array([row['true'] for row in records], dtype=int)
    predicted = np.array([row['predicted'] for row in records], dtype=int)
    matrix = np.asarray(report['confusion'], dtype=int)
    if (report['split'] != 'test' or len(records) != report['count'] or
            not np.array_equal(matrix, confusion_matrix(actual, predicted, labels=range(len(classes))))):
        raise ValueError('Confusion matrix hoặc số lượng không khớp dự đoán test.')
    tp = matrix.diagonal()
    fp = matrix.sum(axis=0) - tp
    fn = matrix.sum(axis=1) - tp
    tn = matrix.sum() - tp - fp - fn

    def divide(numerator, denominator):
        return np.divide(numerator, denominator, out=np.full(len(classes), np.nan), where=denominator != 0)

    table = pd.DataFrame({
        'Lớp': classes, 'Support': tp + fn, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'Precision': divide(tp, tp + fp), 'TPR/Recall': divide(tp, tp + fn),
        'FPR': divide(fp, fp + tn), 'Specificity': divide(tn, fp + tn),
        'F1': divide(2 * tp, 2 * tp + fp + fn),
    })
    # F1 of an absent class follows sklearn zero_division=0 for macro aggregation.
    macro_f1 = np.nan_to_num(table['F1'].to_numpy(), nan=0.0).mean()
    summary = {'Accuracy': float((actual == predicted).mean()), 'Loss': report['loss'],
               'Macro-F1': macro_f1, 'Mean TPR': table['TPR/Recall'].mean(),
               'Mean FPR': table['FPR'].mean(), 'Macro ROC-AUC': np.nan, 'Micro ROC-AUC': np.nan}
    curves = {}
    if 'probabilities' in report:
        scores = np.asarray(report['probabilities'], dtype=float)
        if (report.get('probability_class_order') != classes or
                scores.shape != (len(actual), len(classes)) or
                not np.isfinite(scores).all() or np.any(scores < 0) or np.any(scores > 1) or
                not np.allclose(scores.sum(axis=1), 1, atol=1e-5) or
                not np.array_equal(scores.argmax(axis=1), predicted)):
            raise ValueError('Xác suất test không khớp thứ tự lớp, số ảnh hoặc argmax.')
        binary = np.eye(len(classes), dtype=int)[actual]
        aucs = []
        for label, name in enumerate(classes):
            if len(np.unique(binary[:, label])) < 2:
                aucs.append(np.nan)
                continue
            fpr, tpr, _ = roc_curve(binary[:, label], scores[:, label])
            auc_value = roc_auc_score(binary[:, label], scores[:, label])
            curves[name] = (fpr, tpr, auc_value)
            aucs.append(auc_value)
        table['ROC-AUC OvR'] = aucs
        if np.isfinite(aucs).all():
            summary['Macro ROC-AUC'] = float(np.mean(aucs))
        summary['Micro ROC-AUC'] = roc_auc_score(binary.ravel(), scores.ravel())
        curves['micro'] = (*roc_curve(binary.ravel(), scores.ravel())[:2], summary['Micro ROC-AUC'])
    else:
        table['ROC-AUC OvR'] = np.nan
    return table, summary, curves


test_analyses = []
for report in reports:
    if report['split'] != 'test':
        continue
    if (report['settings']['model_name'] not in config.MODEL_NAMES or
            report['settings']['model_config'] not in config.EXPERIMENTS.values()):
        continue
    table, summary, curves = analyse_test_report(report)
    settings = report['settings']
    label = f"{settings['model_name']} {settings['experiment']} | epoch {report['epoch']} | {report['checkpoint_sha256'][:8]}"
    print(label, '| N =', report['count'], '|', report['evaluated_at'])
    display(table.round(4))
    if not curves:
        print('Báo cáo thiếu xác suất từng lớp: có TPR/FPR, chưa có ROC-AUC.')
    test_analyses.append((label, report, table, summary, curves))
if not test_analyses:
    print('Chưa có báo cáo test của hai model trong ma trận thí nghiệm.')


### 7.2. ROC curve và AUC theo lớp

ROC biểu diễn **TPR theo FPR** khi thay đổi ngưỡng trên xác suất của từng lớp.
Đường chéo là mức xếp hạng ngẫu nhiên (AUC = 0,5); AUC lớn hơn thể hiện khả năng xếp hạng positive trước negative tốt hơn.

- **Per-class OvR AUC:** đánh giá khả năng tách một lớp khỏi phần còn lại.
- **Macro AUC:** trung bình AUC của 10 lớp, mỗi lớp có trọng số bằng nhau.
- **Micro AUC:** gộp các cặp ảnh–lớp rồi tính một đường ROC chung.

ROC/AUC cần toàn bộ xác suất 10 lớp; không suy ra từ confusion matrix, nhãn argmax hoặc confidence cao nhất.
TPR/FPR từ argmax ở bảng trước không phải một ngưỡng xác suất cố định dùng cho ROC OvR.
AUC cao không đồng nghĩa accuracy cao hoặc xác suất được hiệu chuẩn tốt; cần đọc cùng Macro-F1 và kết quả từng lớp.
Lớp không có đủ positive/negative không có AUC xác định; macro AUC chỉ công bố khi đủ cả 10 lớp.


In [ ]:
for label, report, table, summary, curves in test_analyses:
    class_curves = {name: curves[name] for name in report['classes'] if name in curves}
    if not class_curves:
        continue
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, (fpr, tpr, auc_value) in class_curves.items():
        ax.plot(fpr, tpr, label=f'{name}: {auc_value:.4f}')
    ax.plot([0, 1], [0, 1], 'k--', label='Ngẫu nhiên: 0.5')
    ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
           title=label + ' — ROC OvR', xlim=(0, 1), ylim=(0, 1.02))
    ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()


### 7.3. So sánh hai model trên cùng tập test

Bảng tổng hợp đặt accuracy/Macro-F1 cạnh TPR/FPR và ROC-AUC để tránh kết luận chỉ từ một chỉ số.
So sánh trực tiếp yêu cầu cùng test split, thứ tự lớp và bộ ID/nhãn thật; checkpoint phải được chọn trước bằng validation.
Không chọn lại model, learning rate hoặc ngưỡng dự đoán theo các kết quả test này.

Đường macro ROC dùng nội suy TPR trên một lưới FPR chung; nhãn AUC hiển thị trung bình AUC từng lớp.
Đọc thêm bảng từng lớp để xác định lớp nào tạo ra chênh lệch, thay vì chỉ xét điểm trung bình.


In [ ]:
comparison_rows = []
if test_analyses:
    reference = test_analyses[0][1]
    reference_labels = [(r['id'], r['true']) for r in reference['predictions']]
    for label, report, table, summary, curves in test_analyses:
        labels = [(r['id'], r['true']) for r in report['predictions']]
        if (report['classes'] != reference['classes'] or labels != reference_labels or
                report['settings']['split_sha256'] != reference['settings']['split_sha256']):
            raise ValueError('Các báo cáo test không cùng split/lớp/ID/nhãn để so sánh.')
        comparison_rows.append({'Model / checkpoint': label, **summary})
    display(pd.DataFrame(comparison_rows).set_index('Model / checkpoint').round(4))

    with_scores = [entry for entry in test_analyses if entry[4]]
    if with_scores:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        grid = np.linspace(0, 1, 1001)
        for label, report, table, summary, curves in with_scores:
            if np.isfinite(summary['Macro ROC-AUC']):
                mean_tpr = np.mean([np.interp(grid, curves[name][0], curves[name][1])
                                    for name in report['classes']], axis=0)
                axes[0].plot(grid, mean_tpr, label=f"{label}: {summary['Macro ROC-AUC']:.4f}")
            fpr, tpr, auc_value = curves['micro']
            axes[1].plot(fpr, tpr, label=f'{label}: {auc_value:.4f}')
        for ax, title in zip(axes, ('Macro ROC (OvR)', 'Micro ROC (OvR)')):
            ax.plot([0, 1], [0, 1], 'k--')
            ax.set(xlabel='FPR', ylabel='TPR', title=title, xlim=(0, 1), ylim=(0, 1.02))
            ax.legend(fontsize=7); ax.grid(alpha=0.2)
        plt.tight_layout(); plt.show()


In [ ]:
if test_analyses:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for label, report, table, summary, curves in test_analyses:
        for ax, metric in zip(axes, ('TPR/Recall', 'FPR', 'ROC-AUC OvR')):
            ax.plot(table['Lớp'], table[metric], marker='o', label=label)
            ax.set_title(metric); ax.tick_params(axis='x', rotation=60)
            ax.set_ylim(0, 1.02); ax.grid(alpha=0.2)
    for ax in axes:
        ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()


### 7.4. Precision–Recall và Average Precision

PR curve thể hiện precision theo recall khi thay đổi ngưỡng trên xác suất của từng lớp trong thiết lập OvR.
PR bổ sung góc nhìn về chất lượng positive, đặc biệt khi số negative lớn hơn positive: mỗi lớp CIFAR-10 có tỷ lệ 1:9 trên tập test chuẩn.

**Average Precision (AP)** tổng hợp precision tại các mức tăng recall:

$$AP=\sum_n (R_n-R_{n-1})P_n$$

AP ở đây được tính bằng `average_precision_score`, không phải diện tích nội suy hình thang dưới PR curve.
Macro-AP lấy trung bình AP từng lớp; micro-AP gộp tất cả cặp ảnh–lớp.
Mức tham chiếu PR phụ thuộc tỷ lệ positive: mỗi lớp test CIFAR-10 chuẩn có prevalence 0,1; không dùng mức 0,5 của ROC.
Các chỉ số này chỉ mô tả checkpoint đã khóa, không dùng test để chọn lại ngưỡng.


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

pr_summaries = []
micro_pr_curves = []
for label, report, table, summary, roc_curves in test_analyses:
    if not roc_curves:
        continue
    actual = np.array([row['true'] for row in report['predictions']], dtype=int)
    scores = np.asarray(report['probabilities'], dtype=float)
    binary = np.eye(len(report['classes']), dtype=int)[actual]
    ap_rows = []
    fig, ax = plt.subplots(figsize=(7, 5))
    for index, name in enumerate(report['classes']):
        prevalence = float(binary[:, index].mean())
        ap = np.nan
        if 0 < prevalence < 1:
            precision, recall, _ = precision_recall_curve(binary[:, index], scores[:, index])
            ap = average_precision_score(binary[:, index], scores[:, index])
            ax.step(recall, precision, where='post', label=f'{name}: AP={ap:.4f}')
        ap_rows.append({'Lớp': name, 'Positive prevalence': prevalence, 'AP': ap})
    ax.set(xlabel='Recall', ylabel='Precision', title=label + ' — PR OvR',
           xlim=(0, 1), ylim=(0, 1.02))
    ax.legend(fontsize=8, loc='lower left'); ax.grid(alpha=0.2)
    plt.tight_layout(); plt.show()
    ap_table = pd.DataFrame(ap_rows)
    print(label); display(ap_table.round(4))
    macro_ap = ap_table['AP'].mean() if ap_table['AP'].notna().all() else np.nan
    micro_ap = average_precision_score(binary.ravel(), scores.ravel())
    pr_summaries.append({'Model / checkpoint': label, 'Macro-AP': macro_ap, 'Micro-AP': micro_ap})
    precision, recall, _ = precision_recall_curve(binary.ravel(), scores.ravel())
    micro_pr_curves.append((label, recall, precision, micro_ap, binary.mean()))

if pr_summaries:
    display(pd.DataFrame(pr_summaries).set_index('Model / checkpoint').round(4))
    fig, ax = plt.subplots(figsize=(7, 5))
    for label, recall, precision, ap, baseline in micro_pr_curves:
        ax.step(recall, precision, where='post', label=f'{label}: AP={ap:.4f}')
    ax.axhline(micro_pr_curves[0][4], color='gray', linestyle='--', label='Positive prevalence')
    ax.set(xlabel='Recall', ylabel='Precision', title='Micro PR — so sánh model',
           xlim=(0, 1), ylim=(0, 1.02))
    ax.legend(fontsize=8); ax.grid(alpha=0.2); plt.tight_layout(); plt.show()
else:
    print('Chưa có xác suất test để phân tích Precision–Recall/AP.')


### 7.5. Đối chiếu lỗi trên cùng ảnh test

Accuracy trung bình chưa cho biết hai model sai trên những ảnh nào.
Đối chiếu theo cùng ID ảnh và nhãn thật chia kết quả thành bốn nhóm:

| Nhóm | Ý nghĩa |
|---|---|
| Cả hai đúng | Cả A và B phân loại đúng |
| A sai, B đúng | B sửa được lỗi của A |
| A đúng, B sai | B sai ở ảnh A phân loại đúng |
| Cả hai sai | Hai model cùng sai, có thể dự đoán khác lớp |

Chênh lệch accuracy B−A bằng `(số A sai/B đúng − số A đúng/B sai) / tổng số ảnh`.
Bảng theo lớp và ảnh minh họa giúp xác định cải thiện có tập trung ở vài lớp hay không.
Các checkpoint được xác định trước bằng validation; không tự chọn cặp theo điểm test cao nhất.


In [ ]:
def compare_test_errors(report_a, report_b):
    if (report_a['split'] != 'test' or report_b['split'] != 'test' or
            report_a['classes'] != report_b['classes'] or
            report_a['settings']['split_sha256'] != report_b['settings']['split_sha256']):
        raise ValueError('Hai báo cáo không cùng test split hoặc thứ tự lớp.')
    records_a = {row['id']: row for row in report_a['predictions']}
    records_b = {row['id']: row for row in report_b['predictions']}
    if (len(records_a) != report_a['count'] or len(records_b) != report_b['count'] or
            records_a.keys() != records_b.keys()):
        raise ValueError('ID test khác nhau hoặc có ID trùng.')
    rows = []
    for index in sorted(records_a):
        a, b = records_a[index], records_b[index]
        if a['true'] != b['true']:
            raise ValueError('Nhãn thật khác nhau trên cùng ID.')
        correct_a = a['predicted'] == a['true']
        correct_b = b['predicted'] == b['true']
        if correct_a and correct_b:
            group = 'Cả hai đúng'
        elif not correct_a and correct_b:
            group = 'A sai, B đúng'
        elif correct_a and not correct_b:
            group = 'A đúng, B sai'
        else:
            group = 'Cả hai sai'
        rows.append({'ID': index, 'Nhãn thật': report_a['classes'][a['true']],
                     'Dự đoán A': report_a['classes'][a['predicted']],
                     'Dự đoán B': report_b['classes'][b['predicted']],
                     'Confidence A': a.get('confidence', np.nan),
                     'Confidence B': b.get('confidence', np.nan), 'Kết quả': group})
    return pd.DataFrame(rows)


# Hai nhãn checkpoint cố định; None dùng cặp duy nhất nếu mỗi model có đúng một báo cáo.
PAIR_LABELS = None
pair = []
if PAIR_LABELS is not None:
    if len(PAIR_LABELS) != 2 or len(set(PAIR_LABELS)) != 2:
        raise ValueError('PAIR_LABELS cần hai nhãn checkpoint khác nhau.')
    by_label = {entry[0]: entry for entry in test_analyses}
    pair = [by_label[label] for label in PAIR_LABELS]
else:
    candidates = [[entry for entry in test_analyses if entry[1]['settings']['model_name'] == name]
                  for name in config.MODEL_NAMES]
    if all(len(entries) == 1 for entries in candidates):
        pair = [entries[0] for entries in candidates]

paired_errors = None
if pair:
    label_a, report_a = pair[0][:2]
    label_b, report_b = pair[1][:2]
    if report_a['settings']['model_name'] == report_b['settings']['model_name']:
        raise ValueError('Đối chiếu này cần VGG16 và DenseNet121.')
    paired_errors = compare_test_errors(report_a, report_b)
    print('A:', label_a, '| B:', label_b)
    groups = ['Cả hai đúng', 'A sai, B đúng', 'A đúng, B sai', 'Cả hai sai']
    counts = paired_errors['Kết quả'].value_counts().reindex(groups, fill_value=0)
    display(pd.DataFrame({'Số ảnh': counts, 'Tỷ lệ': counts / len(paired_errors)}))
    print('Accuracy B - A:', (counts['A sai, B đúng'] - counts['A đúng, B sai']) / len(paired_errors))
    display(pd.crosstab(paired_errors['Nhãn thật'], paired_errors['Kết quả']).reindex(columns=groups, fill_value=0))
else:
    print('Chưa có cặp checkpoint test xác định duy nhất để đối chiếu lỗi.')


In [ ]:
if paired_errors is not None:
    wrong_groups = ['A sai, B đúng', 'A đúng, B sai', 'Cả hai sai']
    selections = [paired_errors[paired_errors['Kết quả'] == group].head(4) for group in wrong_groups]
    examples = pd.concat(selections, ignore_index=True)
    display(examples)
    test_path = project / 'data' / 'cifar-10-batches-py' / 'test_batch'
    if len(examples) and test_path.exists():
        from torchvision.datasets import CIFAR10
        test_images = CIFAR10(root=str(project / 'data'), train=False, download=False)
        if examples['ID'].min() < 0 or examples['ID'].max() >= len(test_images):
            raise ValueError('ID báo cáo không thuộc dataset test local.')
        rows = (len(examples) + 3) // 4
        fig, axes = plt.subplots(rows, 4, figsize=(14, rows * 3.6), squeeze=False)
        for ax, (_, example) in zip(axes.flat, examples.iterrows()):
            index = int(example['ID'])
            actual_name = test_images.classes[test_images.targets[index]]
            if actual_name != example['Nhãn thật']:
                raise ValueError('Dataset local không khớp nhãn báo cáo.')
            ax.imshow(test_images.data[index])
            ax.set_title(f"ID {index} | {example['Kết quả']}\nThật: {actual_name}\n"
                         f"A: {example['Dự đoán A']} ({example['Confidence A']:.2f})\n"
                         f"B: {example['Dự đoán B']} ({example['Confidence B']:.2f})", fontsize=8)
            ax.axis('off')
        for ax in list(axes.flat)[len(examples):]:
            ax.axis('off')
        plt.tight_layout(); plt.show()
        del test_images
    elif len(examples):
        print('Chưa có dataset test local để hiển thị ảnh; bảng đối chiếu vẫn dùng được.')


### 7.6. Hướng đọc kết quả

- **TPR thấp, FN cao:** model bỏ sót nhiều ảnh của lớp; đối chiếu confusion matrix để biết chúng bị nhận sang lớp nào.
- **FPR cao, FP cao:** model dễ gán lớp đó cho ảnh thuộc các lớp khác; đọc kèm precision.
- **AUC cao nhưng F1/accuracy thấp:** thứ tự score có thể tốt nhưng quyết định phân lớp chưa tốt; không tự suy ra nguyên nhân hay điều chỉnh ngưỡng trên test.
- **Macro tốt nhưng một lớp yếu:** điểm trung bình có thể che lỗi cục bộ; phân tích cặp lớp nhầm và ảnh sai cùng ID.
- **Chênh lệch nhỏ:** một seed và một test set chưa đủ kết luận model vượt trội một cách ổn định.

Các kết luận định lượng chỉ được điền khi có báo cáo test thực của các checkpoint đã khóa.


## 8. Nhận xét và giới hạn

Việc so sánh được tổ chức theo ba hướng: freeze so với fine-tune trong cùng model; hai learning rate khi fine-tune; và VGG16 so với DenseNet121 trong cùng cấu hình.
Kết luận dựa trên validation accuracy/loss, kết quả theo lớp và chi phí huấn luyện của các lượt đã hoàn tất.

- Các lượt chưa đủ epoch chưa cung cấp bằng chứng đầy đủ để xếp hạng model hoặc cấu hình.
- Cùng learning rate không đồng nghĩa với cấu hình tối ưu của mọi kiến trúc.
- Các nhóm cuối có số tham số khác nhau; cần xem xét cả chất lượng dự đoán và chi phí tính toán.
- Thí nghiệm dùng một seed, chưa đánh giá biến thiên giữa nhiều lần chạy.
- Chưa có báo cáo test thì chưa kết luận về chất lượng trên tập test.

### Tài liệu tham khảo

- [Yêu cầu Practice 2](Practice_2_Pretrained_CNN_Requirements.md).
- [Thiết kế thí nghiệm](Experiment_Plan.md).
- [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html).
- [Bài tham khảo LAB2 — cấu trúc trình bày](https://github.com/zombieTDV/Deeplearning_Course_UTH/tree/LAB2).

- [scikit-learn — Multiclass ROC](https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc.html).

- [scikit-learn — Precision–Recall](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html).
